# N068 · 约束满足、数独与N皇后

**目标：** 以约束传播缩小回溯分支并验证完整解。

**先修：** N065, N067, N007。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** 行列/对角线约束；候选集；最少候选优先；位掩码扩展。

## 从问题到算法

约束满足问题的关键是尽早拒绝部分不合法解。N皇后每行只放一个，列与两条对角线集合足以检查冲突。数独采用最少候选格优先，减少分支；这是搜索顺序优化，不改变可解性。求解器必须区分“找到真实解”和“无解”，不能返回填满却违反约束的表格。

## 最小实现

**本章接口：** `solve_n_queens(n)`、`solve_sudoku(board)`

In [1]:
def solve_n_queens(n):
    if n<0: raise ValueError('n must be nonnegative')
    out=[]; columns=set(); diagonal1=set(); diagonal2=set(); chosen=[]
    def visit(row):
        if row==n:
            out.append(['.'*c+'Q'+'.'*(n-c-1) for c in chosen]); return
        for col in range(n):
            if col in columns or row-col in diagonal1 or row+col in diagonal2: continue
            columns.add(col); diagonal1.add(row-col); diagonal2.add(row+col); chosen.append(col)
            visit(row+1)
            chosen.pop(); columns.remove(col); diagonal1.remove(row-col); diagonal2.remove(row+col)
    visit(0); return out

def solve_sudoku(board):
    if len(board)!=9 or any(len(row)!=9 for row in board): raise ValueError('9 by 9 board required')
    rows=[set() for _ in range(9)]; cols=[set() for _ in range(9)]; boxes=[set() for _ in range(9)]; empty=[]
    digits=set('123456789')
    for r in range(9):
        for c in range(9):
            value=board[r][c]; box=(r//3)*3+c//3
            if value=='.': empty.append((r,c)); continue
            if value not in digits: raise ValueError('invalid cell')
            if value in rows[r] or value in cols[c] or value in boxes[box]: return False
            rows[r].add(value); cols[c].add(value); boxes[box].add(value)
    def candidates(r,c): return digits-rows[r]-cols[c]-boxes[(r//3)*3+c//3]
    def visit(position):
        if position==len(empty): return True
        best=min(range(position,len(empty)),key=lambda i:len(candidates(*empty[i])))
        empty[position],empty[best]=empty[best],empty[position]
        r,c=empty[position]; box=(r//3)*3+c//3
        for value in sorted(candidates(r,c)):
            board[r][c]=value; rows[r].add(value); cols[c].add(value); boxes[box].add(value)
            if visit(position+1): return True
            rows[r].remove(value); cols[c].remove(value); boxes[box].remove(value); board[r][c]='.'
        empty[position],empty[best]=empty[best],empty[position]
        return False
    return visit(0)

## 正确性、前提与复杂度

维护集合恰好反映已放置数字或皇后的约束；候选只删去当前已违反约束的分支，因此不漏解。失败时撤销全部本层修改，无解返回后原题保持不变。

**代价：** N皇后最坏O(n!)级搜索；有E个空格的数独粗界O(9ᴱ)，MRV降低实际搜索但不改变指数最坏性质。

## 小实例与可视化

先手算，再运行。下表由当前内核中的代码和显式小实例生成；不是预制答案图片。

In [2]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

solutions=solve_n_queens(4)
show_table(['解编号','棋盘行','布局'],[(i,r,line) for i,b in enumerate(solutions,1) for r,line in enumerate(b)])

解编号,棋盘行,布局
1,0,.Q..
1,1,...Q
1,2,Q...
1,3,..Q.
2,0,..Q.
2,1,Q...
2,2,...Q
2,3,.Q..


## 自动测试

以下断言检查实现契约。测试通过不等同于一般性证明，也不表示已经提交 LeetCode 官方评测。

In [3]:
from copy import deepcopy
assert len(solve_n_queens(4))==2 and len(solve_n_queens(1))==1
for n in range(1,7):
    for board in solve_n_queens(n):
        pos=[row.index('Q') for row in board]
        assert len(set(pos))==n and len({r-c for r,c in enumerate(pos)})==n and len({r+c for r,c in enumerate(pos)})==n
puzzle=['53..7....','6..195...','.98....6.','8...6...3','4..8.3..1','7...2...6','.6....28.','...419..5','....8..79']
b=[list(row) for row in puzzle]; assert solve_sudoku(b)
want=set('123456789')
assert all(set(row)==want for row in b)
assert all({b[r][c] for r in range(9)}==want for c in range(9))
assert all({b[r][c] for r in range(br,br+3) for c in range(bc,bc+3)}==want for br in [0,3,6] for bc in [0,3,6])
assert all(puzzle[r][c]=='.' or puzzle[r][c]==b[r][c] for r in range(9) for c in range(9))
unsolved=[list('.12345678')]+[['.']*9 for _ in range(8)]; unsolved[3][0]='9'; old=deepcopy(unsolved)
assert solve_sudoku(unsolved) is False and unsolved==old
print('N068: 所有本章断言通过')

N068: 所有本章断言通过


## 练习：先独立完成

**练习 1：** 比较固定填格顺序与最少候选优先。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

**练习 2：** 统计搜索节点而非只比运行时间。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

未完成练习不阻断教学区 Run All。验收时解释为什么正确，不只展示运行截图。

## 代表题与迁移

- 51. N-Queens（canonical）
- 37. Sudoku Solver（canonical）

这些题用于知识映射；本章实现遵循上文明确的教学契约。除原规格注明已核对身份的条目外，本轮未重新联网核验全部题面、收费权限或平台语言支持。不要把同概念教学实例当作所有官方题的完整答案。

## 自测与下一步

不看代码，复述状态、不变量和一个失效条件；改变一个输入约束，判断实现是否仍成立。确认能解释边界测试后，按 `specs/curriculum.json` 的 `learning_order` 进入下一章。